# Unified Zeek TSV Preprocessing Notebook

This notebook standardizes preprocessing for Zeek-generated TSV files.

In [1]:
import sys

sys.path.insert(0, '..')

from __future__ import annotations
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from utils.utils import *

In [2]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )

def clean_numeric_columns(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    cols = [col for col in numeric_cols if col in df.columns]

    df[cols] = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )
    return df

def compute_time_elapsed(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["id.orig_h", "id.resp_h", "ts"]).reset_index(drop=True)
    df["time_elapsed"] = df.groupby(["id.orig_h", "id.resp_h"])["ts"].diff().fillna(999999.0)
    return df


def compute_valid_tcp_handshake(df: pd.DataFrame) -> pd.DataFrame:
    proto_ok = df["proto"].eq("tcp")
    history_ok = df["history"].str.contains(
        r"(?=.*S)(?=.*h)(?=.*A)",
        regex=True,
        na=False
    )
    df["valid_tcp_handshake"] = (proto_ok & history_ok).astype(int)
    return df


def compute_valid_http_conn(df: pd.DataFrame) -> pd.DataFrame:
    service = df["service"].astype(str).str.lower()

    proto_ok = df["proto"].eq("tcp")
    http_ok = df["id.resp_p"].isin([80, 8080, 8000]) & service.eq("http")
    https_ok = df["id.resp_p"].isin([443, 8443]) & service.isin(["https", "ssl"])
    has_data = df["history"].str.contains(r"D", regex=True, na=False)

    df["valid_http_conn"] = (proto_ok & (http_ok | https_ok) & has_data).astype(int)
    return df


def compute_rate_features(df: pd.DataFrame) -> pd.DataFrame:
    duration_safe = (
        df["duration"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .mask(df["duration"] <= 0, 1e-6)
    )
    df["orig_pkt_rate"] = df["orig_pkts"] / duration_safe
    df["orig_byte_rate"] = df["orig_bytes"] / duration_safe
    return df


def compute_portscan_window_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    failed_states = {"S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH", "SH", "SHR"}

    df["window_id"] = (df["ts"] // window_seconds).astype(int)
    df["is_failed_conn"] = df["conn_state"].astype(str).isin(failed_states).astype(int)

    agg = (
        df.groupby(["id.orig_h", "window_id"])
        .agg(
            uniq_dst_ports=("id.resp_p", "nunique"),
            total_orig_pkts=("orig_pkts", "sum"),
            scan_duration=("duration", "max"),
            total_flows=("id.orig_h", "size"),
            failed_flows=("is_failed_conn", "sum"),
        )
        .reset_index()
    )

    agg["pkts_per_port"] = (agg["total_orig_pkts"] / agg["uniq_dst_ports"].replace(0, np.nan)).fillna(0.0)
    agg["fail_ratio"] = (agg["failed_flows"] / agg["total_flows"].replace(0, np.nan)).fillna(0.0)

    return df.merge(
        agg[["id.orig_h", "window_id", "uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]],
        on=["id.orig_h", "window_id"],
        how="left",
    )


def add_engineered_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    print_section("Adding engineered features")
    print(f"input rows: {len(df):,}")

    df = clean_numeric_columns(df, MODEL_NUMERIC_FEATURES)
    df = compute_time_elapsed(df)
    df = compute_valid_tcp_handshake(df)
    df = compute_valid_http_conn(df)
    df = compute_rate_features(df)
    df = compute_portscan_window_features(df, window_seconds=window_seconds)
    df = clean_numeric_columns(df, ENGINEERED_FEATURES)

    print(f"output rows: {len(df):,}")
    print("engineered features:", ", ".join(ENGINEERED_FEATURES))
    return df

def drop_invalid_http_flood_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    is_dos = df["label"] == "DOS_HTTP_FLOOD"
    not_tcp = df["proto"].astype(str).str.lower().ne("tcp")
    not_correct_service = ~df["service"].astype(str).str.lower().isin({"http", "https", "ssl"})

    remove_mask = is_dos & ( not_tcp |  not_correct_service)

    removed = remove_mask.sum()
    print(f"Removed {removed} DOS_HTTP_FLOOD rows")

    return df.loc[~remove_mask].reset_index(drop=True)

In [3]:

# def add_ddos_udp_flood_features(
#     df: pd.DataFrame,
#     window_seconds: float = 5.0,
# ) -> pd.DataFrame:
#     df = df.copy()

#     df["is_udp"] = df["proto"].astype(str).str.lower().eq("udp").astype(int)
#     df["window_id"] = (df["ts"] // window_seconds).astype(int)

#     group_cols = ["id.resp_h", "window_id"]
#     udp_df = df[df["is_udp"] == 1]

#     agg = udp_df.groupby(group_cols).agg(
#         udp_conn_count=("id.orig_h", "size"),
#         udp_packets=("orig_pkts", "sum"),
#         unique_src_ips=("id.orig_h", "nunique"),
#     ).reset_index()

#     # Use window duration instead of flow duration
#     agg["udp_rate"] = agg["udp_packets"] / window_seconds

#     df = df.merge(
#         agg,
#         on=["id.resp_h", "window_id"],
#         how="left",
#     )

#     for col in [
#         "udp_conn_count",
#         "udp_packets",
#         "udp_rate",
#         "unique_src_ips",
#     ]:
#         df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

#     return df

# def add_ddos_syn_flood_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
#     df = df.copy()

#     df["window_id"] = (df["ts"] // window_seconds).astype(int)

#     tcp_df = df[df["proto"].astype(str).str.lower() == "tcp"].copy()

#     tcp_df["is_syn"] = tcp_df["history"].astype(str).str.contains("S", regex=False).astype(int)
#     tcp_df["is_ack"] = tcp_df["history"].astype(str).str.contains("A", regex=False).astype(int)

#     group_cols = ["id.resp_h", "window_id"]

#     agg = tcp_df.groupby(group_cols).agg(
#         syn_conn_count=("id.orig_h", "size"),
#         syn_count=("is_syn", "sum"),
#         ack_count=("is_ack", "sum"),
#         source_ip_count=("id.orig_h", "nunique"),
#     ).reset_index()

#     agg["half_open_count"] = (agg["syn_count"] - agg["ack_count"]).clip(lower=0)
#     agg["syn_duration"] = window_seconds
#     agg["syn_rate"] = agg["syn_count"] / window_seconds

#     df = df.merge(
#         agg[
#             [
#                 "id.resp_h",
#                 "window_id",
#                 "syn_duration",
#                 "syn_conn_count",
#                 "syn_count",
#                 "syn_rate",
#                 "half_open_count",
#                 "source_ip_count",
#             ]
#         ],
#         on=["id.resp_h", "window_id"],
#         how="left",
#     )

#     for col in [
#         "syn_duration",
#         "syn_conn_count",
#         "syn_count",
#         "syn_rate",
#         "half_open_count",
#         "source_ip_count",
#     ]:
#         df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

#     return df


In [4]:
def load_and_prepare_dataset(
    datapath: str,
    target_labels: list[str],
    window_seconds: float = 5.0,
) -> pd.DataFrame:
    print(f"Loading data from: {datapath}")
    df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")
    df.columns = df.columns.str.strip()

    print("Raw shape:", df.shape)

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    df["label"] = df["label"].astype(str).map(normalize_label_name)
    target_labels = [normalize_label_name(x) for x in target_labels]
    df = df[df["label"].isin(target_labels)].copy()

    df = drop_invalid_http_flood_rows(df)

    print("Processed shape:", df.shape)
    if "label" in df.columns:
        print(df["label"].value_counts())

    return df

In [5]:
WINDOW_SECONDS = 5.0
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "DDOS_UDP_FLOOD", "DDOS_SYN_FLOOD", "ATTACK"]

def load_cicids2017_data() -> pd.DataFrame:
    print("Loading CICIDS2017 data...")
    df_cicids2017_wednesday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/wednesday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_friday = load_and_prepare_dataset(
        datapath="../data/CICIDS2017/friday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    res = pd.concat([df_cicids2017_wednesday, df_cicids2017_friday], ignore_index=True)
    print("Processed shape:", res.shape)
    print(res["label"].value_counts())
    return res

# def load_cicddos2019_data() -> pd.DataFrame:
#     print("Loading CICIDS2019 data...")
#     df_cicddos2019_1 = load_and_prepare_dataset(
#         datapath="../data/CICDDoS2019/cicddos2019_first_day_1_labeled.tsv",
#         target_labels=TARGET_LABELS,
#         window_seconds=WINDOW_SECONDS
#     )
#     df_cicddos2019_2 = load_and_prepare_dataset(
#         datapath="../data/CICDDoS2019/cicddos2019_first_day_2_labeled.tsv",
#         target_labels=TARGET_LABELS,
#         window_seconds=WINDOW_SECONDS
#     )
#     df_cicddos2019_3 = load_and_prepare_dataset(
#         datapath="../data/CICDDoS2019/cicddos2019_first_day_3_labeled.tsv",
#         target_labels=TARGET_LABELS,
#         window_seconds=WINDOW_SECONDS
#     )
#     df_cicddos2019_4 = load_and_prepare_dataset(
#         datapath="../data/CICDDoS2019/cicddos2019_first_day_4_labeled.tsv",
#         target_labels=TARGET_LABELS,
#         window_seconds=WINDOW_SECONDS
#     )
#     df_cicddos2019_5 = load_and_prepare_dataset(
#         datapath="../data/CICDDoS2019/cicddos2019_second_day_labeled.tsv",
#         target_labels=TARGET_LABELS,
#         window_seconds=WINDOW_SECONDS
#     )
#     res = pd.concat([df_cicddos2019_1, df_cicddos2019_2, df_cicddos2019_3, df_cicddos2019_4, df_cicddos2019_5], ignore_index=True)
#     print("Processed shape:", res.shape)
#     print(res["label"].value_counts())
#     return res

def load_ciciot2023_data() -> pd.DataFrame:
    print("Loading CICIoT2023 data...")
    res = load_and_prepare_dataset(
        datapath="../data/CICIoT2023/ciciot2023_labeled_conn.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

In [15]:
def downsample_label(df: pd.DataFrame, label: str, frac: float) -> pd.DataFrame:
    label_df = df[df["label"] == label]
    df_other = df[df["label"] != label]
    label_df = label_df.sample(frac=frac, random_state=42)
    combined_df = pd.concat([label_df, df_other])
    return combined_df

In [7]:
df_cicids2017 = load_cicids2017_data()

Loading CICIDS2017 data...
Loading data from: ../data/CICIDS2017/wednesday_labeled.tsv
Raw shape: (509362, 23)
Removed 8102 DOS_HTTP_FLOOD rows
Processed shape: (473710, 23)
label
BENIGN            318361
DOS_HTTP_FLOOD    155349
Name: count, dtype: int64
Loading data from: ../data/CICIDS2017/friday_labeled.tsv
Raw shape: (547353, 23)
Removed 0 DOS_HTTP_FLOOD rows
Processed shape: (452004, 23)
label
BENIGN      297970
PORTSCAN    154034
Name: count, dtype: int64
Processed shape: (925714, 23)
label
BENIGN            616331
DOS_HTTP_FLOOD    155349
PORTSCAN          154034
Name: count, dtype: int64


In [8]:
df_cicids2017.to_csv(f"cicids2017_preprocessed.tsv", sep='\t', index=False, header=True)

In [9]:
df_ciciot2023 = load_ciciot2023_data()

Loading CICIoT2023 data...
Loading data from: ../data/CICIoT2023/ciciot2023_labeled_conn.tsv


C:\Users\Rasmus\AppData\Local\Temp\ipykernel_25324\1646106764.py:7: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(datapath, delimiter="\t", on_bad_lines="skip")


Raw shape: (14518511, 23)
Removed 1363138 DOS_HTTP_FLOOD rows
Processed shape: (13155373, 23)
label
DDOS_SYN_FLOOD    7412700
ATTACK            4748328
BENIGN             342255
DDOS_UDP_FLOOD     290106
PORTSCAN           216533
DOS_HTTP_FLOOD     145451
Name: count, dtype: int64
Processed shape: (13155373, 23)
label
DDOS_SYN_FLOOD    7412700
ATTACK            4748328
BENIGN             342255
DDOS_UDP_FLOOD     290106
PORTSCAN           216533
DOS_HTTP_FLOOD     145451
Name: count, dtype: int64


In [10]:
df_ciciot2023.shape

(13155373, 23)

In [ ]:
df_ciciot2023_sampled = downsample_label(df_ciciot2023, label="DDOS_SYN_FLOOD", frac=0.1)
print(df_ciciot2023_sampled["label"].value_counts())
df_ciciot2023_sampled = downsample_label(df_ciciot2023_sampled, label="ATTACK", frac=0.25)
print(df_ciciot2023_sampled["label"].value_counts())

label
ATTACK            4748328
DDOS_SYN_FLOOD     741270
BENIGN             342255
DDOS_UDP_FLOOD     290106
PORTSCAN           216533
DOS_HTTP_FLOOD     145451
Name: count, dtype: int64
label
ATTACK            1187082
DDOS_SYN_FLOOD     741270
BENIGN             342255
DDOS_UDP_FLOOD     290106
PORTSCAN           216533
DOS_HTTP_FLOOD     145451
Name: count, dtype: int64


In [12]:
df_ciciot2023_sampled.to_csv(f"ciciot2023_preprocessed.tsv", sep='\t', index=False, header=True)

# Validation of data

In [13]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
df_ciciot2023_sampled.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label
10390463,1.660680e+09,C7fMiopVzlymuE8e3,192.168.137.134,40839,192.168.137.108,1443,tcp,-,522.798986,43539311,0,OTH,T,T,0,-,19,760,0,0,-,6,ATTACK
11390068,1.660681e+09,CBaqRd4OHjCMyKO17e,192.168.137.195,37096,192.168.137.79,554,tcp,-,673.023548,1810917113,0,OTH,T,T,0,-,34,1360,0,0,-,6,ATTACK
7847463,1.660668e+09,CNHPGX1zutO5kzmp0d,192.168.137.134,55150,192.168.137.123,4070,tcp,-,691.653267,64585601,0,OTH,T,T,0,-,22,880,0,0,-,6,ATTACK
12478779,1.664866e+09,C3sBdSwsYbnIF5tae,fe80::3223:3ff:fef3:57cb,5353,ff02::fb,5353,udp,dns,-,-,-,S0,T,F,0,D,1,318,0,0,-,17,ATTACK
9137833,1.660676e+09,CUIBPlo2DonNce0e1,192.168.137.134,4662,192.168.137.99,55442,tcp,-,417.993256,1374246726,0,OTH,T,T,0,-,14,560,0,0,-,6,ATTACK


# Create small datasets

In [16]:
def downsample_labels(df, TARGET_LABELS, frac=0.1):
    for label in TARGET_LABELS:
        df = downsample_label(df, label=label, frac=frac)
    print(df["label"].value_counts())
    return df

cicids_small = downsample_labels(df_cicids2017, TARGET_LABELS, frac=0.1)
cicids_small.to_csv(f"cicids2017_preprocessed_small.tsv", sep="\t")
ciciot_small = downsample_labels(df_ciciot2023_sampled, TARGET_LABELS, frac=0.1)
ciciot_small.to_csv(f"ciciot2023_preprocessed_small.tsv", sep="\t")

label
BENIGN            61633
DOS_HTTP_FLOOD    15535
PORTSCAN          15403
Name: count, dtype: int64
label
ATTACK            118708
DDOS_SYN_FLOOD     74127
BENIGN             34226
DDOS_UDP_FLOOD     29011
PORTSCAN           21653
DOS_HTTP_FLOOD     14545
Name: count, dtype: int64
